# 🤖 CEM4644 · MP5 — One model for everything? A vision-language model for classification, detection and take-off
## Homework (individual): *Architectural styles, machinery, scanned floor plans*

**No coding needed.** Each grey box below is one *step*: click the ▶ (play) button at its left, wait until it finishes, look at the result, then answer the report question that follows. Run the steps **from top to bottom**.

In MP2, MP3 and MP4 you trained or used one **specialist** model per task: a classifier, a detector, a segmentation model. This lab gives all three tasks to one **generalist**: a large multimodal language model (Gemini), which has never seen our photos and is steered only by the words you send it. The lab is built around two questions: *can a prompt replace a trained model?* and *how do you get an answer a program can read?*

**What you will do (about 120 minutes)**
1. Talk to the model about a photo, and get the answer as JSON in two ways: by asking nicely, and by enforcing a schema.
2. Classify the MP2 photos with three prompts and compare with the MP2 model.
3. Detect and count on the MP3 photos, compare with the MP3 model.
4. Measure rooms from the model's own polygons: one plan through the chat, then every plan at once through the API.
5. Your own image and your own words in a small app.

**Two routes, nothing done twice.** For the *single-example* steps (a photo described, a photo classified, boxes on one site photo, a count, the rooms of one plan) you use a plain chat window: **hokie.ai** (https://hokie.ai.vt.edu/, Virginia Tech's free access to GPT models, sign in with your VT account). You download the example image, paste the prompt the notebook gives you, attach the image, and paste the reply back into the notebook, which draws and scores it against the answer key and next to the earlier labs' specialist models. The *batch* steps (every photo scored at once, prompts compared) use the Gemini API, whose answers are precomputed. No GPU is needed in this notebook.

**Before you start**
- **Gemini API key (free).** Open https://aistudio.google.com/apikey, sign in with your Google account, create a key. In Colab, click the **key icon** in the left bar (*Secrets*), add a secret named `GEMINI_API_KEY` with the key as its value, and switch on *Notebook access*. Without a key the precomputed answers of the built-in examples still work; your own prompts and images do not.

- Never paste your key into a cell you might share: use the secret.

In [ ]:
#@title ▶ Step 0 · Run me first (1–3 minutes) { display-mode: "form" }
#@markdown Click ▶ and wait for the green ✅ line. This downloads the examples with their precomputed answers.
#@markdown Leave *api_key* empty to use the Colab secret GEMINI_API_KEY (recommended).
api_key = "" #@param {type:"string"}
model = "gemini-3.5-flash-lite" #@param ["gemini-3.5-flash-lite", "gemini-3.5-flash", "gemini-3.8-flash", "gemini-3.6-flash"]
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp5_llm_vision", "aec_llm"
FOLDERS = ["mp5_llm_vision"]  # only this lab folder is downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)     # also trims a full copy left by an earlier run
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
for _m in [m for m in list(sys.modules) if m.split(".")[0] in (PKG, "aec_seg")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_llm import lab
lab.setup(dataset="homework", api_key=api_key, model=model, load_sam=False)


## Part 1 · Talk to the model

A **vision-language model** reads an image and text together and answers in text. It has no fixed list of classes and no output layer for boxes: whatever structure you want back, you must **ask for it in words**, and the reply is a piece of text that a program then has to read. That is the whole difference from the specialists: the prompt is the program.

Two things to watch in every step: the **reply itself** (is it what you asked for, is it right?) and its **cost**: seconds per request and **tokens** (the units the service bills; an image costs a few hundred tokens, the model's private *thinking* costs more).

In [ ]:
#@title ▶ Step 1a · The examples and their answer keys { display-mode: "form" }
#@markdown The same kind of material as in MP2, MP3 and MP4, with the answer keys and with what the earlier labs' specialist models said about them.
which = "photos" #@param ["photos", "site photos", "plans"]
lab.show_examples(which)


In [ ]:
#@title ▶ Step 1b · Ask the chat anything about a photo { display-mode: "form" }
#@markdown Free text in, free text out, through the chat window: download the photo, paste the prompt with the photo attached, paste the reply back.
photo = "bauhaus_international_1  (truth: Bauhaus International)" #@param ["bauhaus_international_1  (truth: Bauhaus International)", "bauhaus_international_2  (truth: Bauhaus International)", "brutalist_1  (truth: Brutalist)", "brutalist_2  (truth: Brutalist)", "art_deco_1  (truth: Art Deco)", "art_deco_2  (truth: Art Deco)", "neoclassical_1  (truth: Neoclassical)", "neoclassical_2  (truth: Neoclassical)", "gothic_revival_1  (truth: Gothic Revival)", "gothic_revival_2  (truth: Gothic Revival)", "georgian_1  (truth: Georgian)", "georgian_2  (truth: Georgian)", "victorian_terrace_1  (truth: Victorian Terrace)", "victorian_terrace_2  (truth: Victorian Terrace)", "mid_century_modern_1  (truth: Mid-century Modern)", "mid_century_modern_2  (truth: Mid-century Modern)", "contemporary_curtain_wall_1  (truth: Contemporary Curtain Wall)", "contemporary_curtain_wall_2  (truth: Contemporary Curtain Wall)", "industrial_warehouse_1  (truth: Industrial Warehouse)", "industrial_warehouse_2  (truth: Industrial Warehouse)"]
question = "What do you see in this photo? Answer in three sentences." #@param ["What do you see in this photo? Answer in three sentences.", "Is there anything a building inspector should worry about here? Answer in two sentences.", "Describe this photo as a JSON object with the keys \"what\", \"condition\" and \"action\"."] {allow-input: true}
lab.chat_describe(photo, question)


In [ ]:
#@title ▶ Step 1c · JSON, way A: ask the chat nicely { display-mode: "form" }
#@markdown The classification prompt asks for JSON; nothing enforces it. Paste the reply back: is it valid JSON, is the label one of the categories, is it right? Do it two or three times (new chat each time) and watch whether the format and the label stay the same.
photo = "bauhaus_international_1  (truth: Bauhaus International)" #@param ["bauhaus_international_1  (truth: Bauhaus International)", "bauhaus_international_2  (truth: Bauhaus International)", "brutalist_1  (truth: Brutalist)", "brutalist_2  (truth: Brutalist)", "art_deco_1  (truth: Art Deco)", "art_deco_2  (truth: Art Deco)", "neoclassical_1  (truth: Neoclassical)", "neoclassical_2  (truth: Neoclassical)", "gothic_revival_1  (truth: Gothic Revival)", "gothic_revival_2  (truth: Gothic Revival)", "georgian_1  (truth: Georgian)", "georgian_2  (truth: Georgian)", "victorian_terrace_1  (truth: Victorian Terrace)", "victorian_terrace_2  (truth: Victorian Terrace)", "mid_century_modern_1  (truth: Mid-century Modern)", "mid_century_modern_2  (truth: Mid-century Modern)", "contemporary_curtain_wall_1  (truth: Contemporary Curtain Wall)", "contemporary_curtain_wall_2  (truth: Contemporary Curtain Wall)", "industrial_warehouse_1  (truth: Industrial Warehouse)", "industrial_warehouse_2  (truth: Industrial Warehouse)"]
lab.chat_classify(photo)


> ### 📝 Report question 1
> From Step 1c: across your tries, how many of the chat replies were valid JSON, was the label always one of the categories, and did it stay the same? In Step 2a, run the *basic* prompt with the schema off and on: what does the schema change in the replies, and what does it not change (the label can still be wrong)? Why does a program that has to read the reply (to fill a table, to count, to draw a box) need the schema rather than a polite request?

## Part 2 · Classification by prompt

The MP2 task again: architectural styles of building façades (computer-generated images), 10 classes, and a model that was never trained on them. Three prompts of increasing length are prepared: the class names only, the names with a one-line description each, and the descriptions plus decision rules. The reply is scored against the answer key exactly as in MP2, and the MP2 course model's accuracy on the same photos is shown next to it.

In [ ]:
#@title ▶ Step 2a · Classify all the photos { display-mode: "form" }
#@markdown All three prompts are precomputed with the schema on; *basic* is also precomputed with the schema off. Other combinations run live (a few minutes on the free tier).
prompt = "basic" #@param ["basic", "with descriptions", "with descriptions and rules"]
schema = True #@param {type:"boolean"}
show_mistakes = True #@param {type:"boolean"}
lab.classify(prompt, schema, show_mistakes)


In [ ]:
#@title ▶ Step 2b · Your own prompt { display-mode: "form" }
#@markdown Edit the text (keep `{classes}` where the list of categories should go; `{intro}` is the first sentence). Runs live on every photo: needs your key; the free tier allows only a few requests per minute, so this takes one to three minutes.
prompt_text = "{intro} Classify it into exactly one of these categories: {classes}. Reply with JSON only, no other text, in this form: {\"label\": <one category, spelled exactly as in the list>, \"confidence\": <a number from 0 to 1>, \"reason\": <one short sentence>}" #@param {type:"string"}
schema = True #@param {type:"boolean"}
lab.classify_own(prompt_text, schema)


> ### 📝 Report question 2
> From Step 2a: the accuracy of the three prompts and of the MP2 model on the 20 style images. The images are computer-generated and the MP2 model was trained on the same kind of images: is that a fair comparison? Which styles does Gemini confuse with each other, and does the rules prompt help?

> ### 📝 Report question 3
> From Step 2b: what did you change in the prompt and what accuracy did you get? If it went up, what is the risk of tuning a prompt on the same photos you score it on (think of MP2's training / test split)?

## Part 3 · Detection and counting by prompt

The MP3 task: construction machinery: excavators, dump trucks, wheel loaders. The prompt asks for a list of boxes as `[ymin, xmin, ymax, xmax]` on a 0–1000 grid (the convention this model was trained with), the notebook converts them to pixels and scores them like MP3 did: a box is *found* when its label is right and it overlaps the answer key's box by at least half. The MP3 YOLO model's boxes on the same photos are shown next to Gemini's.

In [ ]:
#@title ▶ Step 3a · Boxes on one photo, from the chat { display-mode: "form" }
#@markdown Paste the reply back and the notebook draws the boxes on the photo and scores them against the answer key, next to the MP3 model's numbers on the same photo. If the boxes land in the wrong place, the chat used another coordinate convention: change *box order* or *numbers are* and score again.
site = "site_1  (1 boxes in the answer key)" #@param ["site_1  (1 boxes in the answer key)", "site_2  (1 boxes in the answer key)", "site_3  (2 boxes in the answer key)", "site_4  (1 boxes in the answer key)", "site_5  (2 boxes in the answer key)", "site_6  (2 boxes in the answer key)"]
lab.chat_detect(site)


In [ ]:
#@title ▶ Step 3b · All the photos, scored { display-mode: "form" }
schema = True #@param {type:"boolean"}
lab.detect_all(schema)


In [ ]:
#@title ▶ Step 3c · Just ask the chat for the number { display-mode: "form" }
#@markdown Instead of boxes, the chat is asked for a count. Compared with the answer key and with the boxes it gave you in Step 3a.
site = "site_1  (1 boxes in the answer key)" #@param ["site_1  (1 boxes in the answer key)", "site_2  (1 boxes in the answer key)", "site_3  (2 boxes in the answer key)", "site_4  (1 boxes in the answer key)", "site_5  (2 boxes in the answer key)", "site_6  (2 boxes in the answer key)"]
lab.chat_count(site)


> ### 📝 Report question 4
> From Step 3b: Gemini's recall and precision against the MP3 YOLO model's on the machinery photos. Which machine is hardest and why? From Step 3c: does the model's count of excavators agree with its own boxes and with the answer key? Also: on the photo you gave the chat in Step 3a, how did its boxes compare with the MP3 model's on the same photo, and what coordinate convention did the chat use?

> ### 📝 Report question 5
> From Step 3c on two photos: the chat's count, the number of boxes it gave you in Step 3a, and the answer key. When they disagree, which one is wrong and how would you know on a site where there is no answer key? Which of the two ways of counting would you trust on a site camera, and why?

## Part 4 · Rooms on a floor plan

The MP4 task on four of the MP4 homework plans: scanned real drawings with furniture and dimension strings (one of them a two-storey sheet), with the same answer keys. The prompt asks for each room's outline as a polygon (a list of points on the 0–1000 grid) and its label; the notebook converts the polygon to pixels and to square metres with the plan's scale, and checks every room against the drawing's answer key, next to MP4's result (SAM 3 asked for *room* by phrase). First one plan by hand in the chat window, then all of them at once through the API.

In [ ]:
#@title ▶ Step 4a · Rooms from the chat's polygons (one plan) { display-mode: "form" }
#@markdown Paste the reply back: the notebook converts the polygons, measures every room in square metres and checks each against the drawing. Long lists of coordinates are where chat replies break: look at what you get, and try a second plan.
plan = "8138: apartment with bold walls and furniture" #@param ["8138: apartment with bold walls and furniture", "10715: apartment (bold walls)", "11615: small flat A3 with printed room areas", "5018: two-storey house: ground floor (left) and upper floor (right)"]
lab.chat_rooms(plan)


In [ ]:
#@title ▶ Step 4b · All 4 plans at once (the API) { display-mode: "form" }
#@markdown The same prompt sent to the API for all 4 plans, which would take you 4 rounds of copying by hand. One picture and one line per plan: rooms found, median error against the drawing, the model's total against the floor area, and MP4's SAM 3 for comparison.
#@markdown Untick *schema* to see what the same prompt returns when nothing enforces the structure (precomputed).
schema = True #@param {type:"boolean"}
lab.segment_all(schema)


> ### 📝 Report question 6
> From Step 4a on plan 8138 and on plan 11615 (scanned drawings with furniture and dimension strings): copy the per-room tables from your pasted replies. What does the scan's clutter do to the chat model's polygons? Then from Step 4b: the table for all four plans from the API, including the two-storey sheet 5018. Where is the API's batch result better or worse than your chat replies, and how does either compare with MP4's SAM 3 by phrase?

## Part 5 · Your image, your words

A small app, opened from a link: pick a built-in image or upload your own, choose a task (it fills in a starting prompt), edit the words, switch the schema on or off, and read the raw reply next to what the notebook draws from it. Needs your key: everything here is live.

In [ ]:
#@title ▶ Step 5 · Prompt lab { display-mode: "form" }
#@markdown This cell prints a **link**: open it in a new tab (or on your phone). The app stays alive while this notebook is running; if no link appears, run the cell again.
lab.prompt_app()


> ### 📝 Report question 7
> Run at least 3 experiments of your own in Step 5 (a photo from a site or from the internet, a plan, a changed prompt, the schema on and off). For each: the image, the prompt, the raw reply, and whether it was right. What kind of request broke the model, and how did it break (wrong answer, invented objects, unreadable reply)?

## Wrap-up · Generalist or specialist?

In [ ]:
#@title ▶ Step 6 · All tasks side by side { display-mode: "form" }
#@markdown One table: the generalist with one prompt per task against the three specialists from MP2, MP3 and MP4, with time and tokens.
lab.summary(chat=True)


> ### 📝 Report question 8
> From Step 6: for each task, would you use the generalist, the specialist, or both together? Argue with the numbers you got and with what each needs: labelled data, training, a GPU, a network connection, money per request, and someone who checks. What does structured output guarantee about a reply, and what does it not guarantee?

In [ ]:
#@title ▶ Numbers for your report { display-mode: "form" }
lab.report_summary()


### Credits
- Photos: Jonathandav/facade-styles (MIT licence; computer-generated images, no real building), the test images of MP2.
- Site photos: Roboflow 100 'excavators' (CC-BY-4.0), the test photos of MP3.
- Floor plans: CubiCasa5K (CC BY-NC-SA 4.0), prepared for MP4 (plans 8138, 10715, 11615, 5018).
- Models: the chat model behind hokie.ai (Virginia Tech) for the single examples; Gemini (Google) through the Gemini API, free tier, for the batch steps.
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp5_llm_vision`).